# Predicting Experimental Semiconductor Band Gaps from Chemical Formula

This notebook introduces a simple materials informatics workflow for predicting experimental semiconductor band gaps from chemical formula. The goal is not to build a perfect model. The goal is to learn how a chemical formula can be converted into numerical descriptors, used to train machine learning models, and interpreted critically.

We will use a public composition-only experimental band gap dataset from `matminer`, parse formulas with `pymatgen`, generate Magpie composition features with `matminer`, and compare a simple baseline model with a random forest model.

## 0. Google Colab Setup

If you opened this notebook in Google Colab, run the next cell before continuing. It installs the packages that are usually missing from a fresh Colab session. If you are running locally from the repository environment, the cell will simply tell you that no Colab setup is needed.

In [ ]:
# This setup cell keeps the notebook runnable in Google Colab.
# It installs the same core packages listed in requirements.txt when Colab is detected.
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab detected. Installing teaching-module dependencies...")
    packages = [
        "pandas>=2.0",
        "numpy>=1.24",
        "matplotlib>=3.7",
        "scikit-learn>=1.3",
        "pymatgen>=2024.5",
        "matminer>=0.9",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    print("Installation complete. Continue with the next cell.")
else:
    print("Not running in Google Colab. Use requirements.txt for local setup.")

## 1. Import Required Packages

This section imports the Python libraries used in the notebook. The comments explain the role of each package.

In [ ]:
# pandas stores tabular data in DataFrames.
import pandas as pd

# numpy provides numerical tools and represents missing values as np.nan.
import numpy as np

# matplotlib creates the plots used to evaluate model predictions.
import matplotlib.pyplot as plt

# pathlib helps save figures using paths that work on different operating systems.
from pathlib import Path

# matminer provides public datasets and materials featurizers.
from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty

# pymatgen understands chemical formulas and compositions.
from pymatgen.core import Composition

# scikit-learn provides train/test splitting, models, and metrics.
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# Show matplotlib plots inside the notebook.
%matplotlib inline

# Use a fixed random seed so the train/test split and random forest are reproducible.
RANDOM_STATE = 42

## 2. Load a Public Experimental Band Gap Dataset

The preferred dataset is `matbench_expt_gap`, a Matbench dataset for predicting experimental band gap from composition alone. If that dataset is not available in a particular matminer installation, the code tries closely related experimental gap datasets as fallbacks.

In [ ]:
# Try the preferred composition-only experimental band gap dataset first.
dataset_candidates = ["matbench_expt_gap", "expt_gap", "expt_gap_kingsbury"]

raw_data = None
dataset_name = None
load_errors = []

for candidate in dataset_candidates:
    try:
        raw_data = load_dataset(candidate)
        dataset_name = candidate
        break
    except Exception as exc:
        load_errors.append(f"{candidate}: {exc}")

if raw_data is None:
    raise RuntimeError("Could not load an experimental band gap dataset. Errors: " + " | ".join(load_errors))

print(f"Loaded dataset: {dataset_name}")
print(f"Number of rows: {len(raw_data):,}")

## 3. Inspect the First Rows and Column Names

Before modeling, always inspect the dataset. We need to know which column contains the chemical formula or composition and which column contains the experimental band gap.

In [ ]:
# Print column names so we can identify the input and target columns.
print("Column names:")
print(list(raw_data.columns))

# Display the first few rows of the dataset.
raw_data.head()

## 4. Identify the Composition and Target Columns

Different datasets sometimes use different column names. The helper function below searches for likely formula/composition column names and likely band gap target column names. For `matbench_expt_gap`, the expected columns are usually `composition` and `gap expt`.

In [ ]:
def find_likely_column(columns, exact_candidates, keyword_groups, description):
    """Find a likely column using exact names first, then keyword groups."""
    normalized = {str(col).strip().lower(): col for col in columns}

    # First try exact matches such as "composition" or "gap expt".
    for candidate in exact_candidates:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]

    # Then try keyword groups such as columns containing both "gap" and "expt".
    for keywords in keyword_groups:
        for col in columns:
            col_text = str(col).strip().lower()
            if all(keyword in col_text for keyword in keywords):
                return col

    raise ValueError(f"Could not identify the {description} column. Available columns: {list(columns)}")


formula_col = find_likely_column(
    raw_data.columns,
    exact_candidates=["composition", "formula", "pretty_formula", "reduced_formula", "chemical_formula"],
    keyword_groups=[["composition"], ["formula"]],
    description="formula/composition",
)

target_col = find_likely_column(
    raw_data.columns,
    exact_candidates=["gap expt", "expt_gap", "band_gap", "band gap", "bandgap", "gap", "egap", "Egap"],
    keyword_groups=[["gap", "expt"], ["band", "gap"], ["gap"]],
    description="experimental band gap target",
)

print(f"Formula/composition column: {formula_col}")
print(f"Band gap target column: {target_col}")

raw_data[[formula_col, target_col]].head()

## 5. Convert Formulas to pymatgen Composition Objects

Machine learning models cannot directly learn from strings like `TiO2` or `CsPbBr3`. First, we convert each formula into a `pymatgen` `Composition` object. A `Composition` stores the elements and stoichiometry in a chemically meaningful form.

In [ ]:
def to_composition(value):
    """Convert a formula-like value to a pymatgen Composition object."""
    try:
        if isinstance(value, Composition):
            return value
        if value is None:
            return np.nan
        return Composition(str(value))
    except Exception:
        return np.nan


# Keep only the useful columns and give the target a clear, unit-aware name.
data = raw_data[[formula_col, target_col]].copy()
data = data.rename(columns={target_col: "band_gap_eV"})

# Convert target values to numbers. Non-numeric entries become missing values.
data["band_gap_eV"] = pd.to_numeric(data["band_gap_eV"], errors="coerce")

# Convert formulas or composition strings into pymatgen Composition objects.
data["composition"] = data[formula_col].apply(to_composition)

# Store a clean reduced formula string for display and error analysis.
data["formula"] = data["composition"].apply(
    lambda comp: comp.reduced_formula if isinstance(comp, Composition) else np.nan
)

# Drop rows that failed formula parsing or have missing/invalid target values.
data = data.dropna(subset=["composition", "formula", "band_gap_eV"]).copy()
data = data[data["band_gap_eV"] >= 0].reset_index(drop=True)

print(f"Rows after formula parsing and target cleaning: {len(data):,}")
data[["formula", "composition", "band_gap_eV"]].head()

### Example Compositions

The examples below show how `pymatgen` interprets several familiar semiconductor or optoelectronic formulas.

In [ ]:
# These examples provide chemical anchors for the rest of the notebook.
example_formulas = ["Si", "GaAs", "TiO2", "PbS", "CdTe", "CsPbBr3"]

example_rows = []
for formula in example_formulas:
    comp = Composition(formula)
    example_rows.append(
        {
            "formula": formula,
            "reduced_formula": comp.reduced_formula,
            "elements": ", ".join(str(element) for element in comp.elements),
            "amounts": comp.get_el_amt_dict(),
        }
    )

pd.DataFrame(example_rows)

## 6. Create Magpie Composition Features with matminer

A feature is a number that a machine learning model can use as input. The `ElementProperty.from_preset("magpie")` featurizer calculates statistics of elemental properties, such as averages, ranges, and modes of properties across the elements in each composition.

In [ ]:
# Use the full dataset by default. For a faster classroom demo, set MAX_ROWS to 1500 or another smaller number.
MAX_ROWS = None

model_data = data[["formula", "composition", "band_gap_eV"]].copy()
if MAX_ROWS is not None and MAX_ROWS < len(model_data):
    model_data = model_data.sample(n=MAX_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"Rows selected for featurization: {len(model_data):,}")

In [ ]:
# Create the Magpie elemental-property featurizer.
featurizer = ElementProperty.from_preset("magpie")

# Single-process featurization is often easier to debug on teaching machines.
try:
    featurizer.set_n_jobs(1)
except AttributeError:
    pass

# Calculate numerical features from the pymatgen Composition objects.
# ignore_errors=True keeps the notebook running if a composition cannot be featurized.
featurized = featurizer.featurize_dataframe(
    model_data,
    col_id="composition",
    ignore_errors=True,
)

print(f"Shape after featurization: {featurized.shape}")
featurized.head()

## 7. Drop Failed Featurizations and Missing Values

A failed featurization usually produces missing feature values. The code below keeps only rows with a known target value and complete numeric Magpie features. This makes the modeling section beginner-friendly because the models receive a clean numerical table.

In [ ]:
# The metadata and target columns are not input features.
metadata_cols = {"formula", "composition", "band_gap_eV"}

# Select numeric feature columns produced by the Magpie featurizer.
feature_cols = [
    col
    for col in featurized.columns
    if col not in metadata_cols and pd.api.types.is_numeric_dtype(featurized[col])
]

# Replace infinite values with missing values so they can be removed cleanly.
featurized[feature_cols] = featurized[feature_cols].replace([np.inf, -np.inf], np.nan)

# Drop rows with missing target values or missing feature values.
rows_before = len(featurized)
clean_data = featurized.dropna(subset=["band_gap_eV"] + feature_cols).reset_index(drop=True)
rows_after = len(clean_data)

print(f"Rows before dropping missing values: {rows_before:,}")
print(f"Rows after dropping missing values: {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Number of Magpie features: {len(feature_cols)}")

clean_data[["formula", "band_gap_eV"] + feature_cols[:5]].head()

## 8. Split into Train and Test Data

The model learns from the training set. The test set is held out until evaluation so we can estimate how well the model predicts materials it did not train on.

In [ ]:
# X contains numerical composition features. y contains experimental band gaps.
X = clean_data[feature_cols]
y = clean_data["band_gap_eV"]
metadata = clean_data[["formula"]]

# Split the features, target values, and formula metadata together.
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

## 9. Train a DummyRegressor Baseline

A baseline model gives us a simple reference point. This `DummyRegressor` ignores the features and predicts the mean training-set band gap for every test material. A useful model should beat this baseline.

In [ ]:
# Train a baseline model that always predicts the mean band gap from the training set.
baseline_model = DummyRegressor(strategy="mean")
baseline_model.fit(X_train, y_train)

# Predict band gaps for the held-out test set.
baseline_pred = baseline_model.predict(X_test)

print("Baseline model trained.")

## 10. Train a RandomForestRegressor

A random forest combines many decision trees. It can learn nonlinear relationships between Magpie composition features and experimental band gaps.

In [ ]:
# Train a random forest model on the same training data.
rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_features="sqrt",
)

rf_model.fit(X_train, y_train)

# Predict band gaps for the held-out test set.
rf_pred = rf_model.predict(X_test)

print("Random forest model trained.")

## 11. Report MAE and R2 for Both Models

Mean absolute error (MAE) reports the average absolute prediction error in electronvolts. R2 compares the model to predicting the mean: values closer to 1 are better, while values near 0 or below indicate weak predictive performance.

In [ ]:
def regression_metrics(y_true, y_pred, model_name):
    """Calculate common regression metrics for a model."""
    return {
        "model": model_name,
        "MAE_eV": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


metrics = pd.DataFrame(
    [
        regression_metrics(y_test, baseline_pred, "DummyRegressor baseline"),
        regression_metrics(y_test, rf_pred, "RandomForestRegressor"),
    ]
)

# Round for easier reading in class discussions.
metrics.round({"MAE_eV": 3, "R2": 3})

## 12. Plot Predicted vs. Actual Band Gap

A predicted-vs-actual plot shows how close the random forest predictions are to the experimental values. Perfect predictions would fall exactly on the diagonal parity line.

In [ ]:
# Save figures in the project-level figures directory when possible.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
figure_dir = project_root / "figures"
figure_dir.mkdir(exist_ok=True)

# Create the parity plot.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, rf_pred, alpha=0.65, s=35, color="#4C78A8", edgecolor="white", linewidth=0.4)

# Plot the perfect-prediction line.
min_gap = min(y_test.min(), rf_pred.min())
max_gap = max(y_test.max(), rf_pred.max())
ax.plot([min_gap, max_gap], [min_gap, max_gap], color="#F58518", linestyle="--", label="Perfect prediction")

ax.set_xlabel("Experimental band gap (eV)")
ax.set_ylabel("Predicted band gap (eV)")
ax.set_title("Random forest: predicted vs. actual band gaps")
ax.legend()
ax.set_aspect("equal", adjustable="box")

pred_plot_path = figure_dir / "predicted_vs_actual_bandgaps.png"
fig.savefig(pred_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {pred_plot_path}")

## 13. Plot the Top 15 Feature Importances

Random forests estimate how much each feature helps reduce prediction error across the trees. Feature importance is useful for interpretation, but it is not proof that a feature directly causes the band gap.

In [ ]:
# Pair each feature name with its random forest importance value.
importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": rf_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

# Select and plot the top 15 most important features.
top15 = importance_df.head(15).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top15["feature"], top15["importance"], color="#54A24B")
ax.set_xlabel("Feature importance")
ax.set_title("Top 15 random forest feature importances")

importance_plot_path = figure_dir / "top15_feature_importances.png"
fig.savefig(importance_plot_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved figure to: {importance_plot_path}")
importance_df.head(15)

## 14. Show the 10 Largest Absolute Errors

The largest errors are often the most interesting cases. They may point to unusual chemistry, missing structural information, polymorphism, defects, measurement differences, or limitations of a composition-only model.

In [ ]:
# Build an error table for the test set.
error_table = meta_test.copy().reset_index(drop=True)
error_table["actual_gap_eV"] = y_test.to_numpy()
error_table["predicted_gap_eV"] = rf_pred
error_table["absolute_error_eV"] = np.abs(error_table["predicted_gap_eV"] - error_table["actual_gap_eV"])

# Show only the columns requested for error analysis.
largest_errors = error_table.sort_values("absolute_error_eV", ascending=False)
largest_errors[["formula", "actual_gap_eV", "predicted_gap_eV", "absolute_error_eV"]].head(10).round(3)

## Reflection Questions

1. Did the random forest beat the baseline? Use MAE and R2 to support your answer.
2. Which feature importances seem chemically meaningful? Which are harder to interpret?
3. Pick one material with a large error. What information is missing from a formula-only model?
4. How could crystal structure, polymorphism, defects, synthesis conditions, or measurement method change the experimental band gap?
5. What additional data would you want before using a model like this for real materials screening?